# Beyond counterfactuals

What else CtreeX explains about a decision tree,
using the [UCI German Credit][gc] data (see `02_german_credit.ipynb`):

1. targets and rankings that use the training statistics of the leaves;
2. what guarantees a prediction -- supportive explanations, invariance ranges
   and contextualised counterfactuals;
3. ordered features, direction constraints and monotonicity audits;
4. feature audits -- where a feature alone decides the outcome;
5. instance-contrastive explanations;
6. data points of counterfactuals -- the nearest one, random samples,
   hidden thresholds and exemplars; and
7. plain-text explanations.

[gc]: https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data

In [1]:
import hashlib, os, urllib.request
import numpy as np

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
SHA256 = 'b21f3d81db8071257d5ff1deaeba1fd4303b62712e6fcc9715c7a86202cb5871'
PATH = 'german.data'
if not os.path.isfile(PATH):
    urllib.request.urlretrieve(URL, PATH)
with open(PATH, 'rb') as f:
    assert hashlib.sha256(f.read()).hexdigest() == SHA256, 'unexpected data file'
rows = [line.split() for line in open(PATH) if line.strip()]
len(rows), rows[0]

(1000,
 ['A11',
  '6',
  'A34',
  'A43',
  '1169',
  'A65',
  'A75',
  '4',
  'A93',
  'A101',
  '4',
  'A121',
  '67',
  'A143',
  'A152',
  '2',
  'A173',
  '1',
  'A192',
  'A201',
  '1'])

In [2]:
FEATURES = [
    ('checking_account', {'A11': '< 0 DM', 'A12': '0 to 200 DM', 'A13': '>= 200 DM', 'A14': 'no checking account'}),
    ('duration', None),
    ('credit_history', {'A30': 'no credits taken', 'A31': 'all credits at this bank paid back',
                        'A32': 'existing credits paid back so far', 'A33': 'delays in the past',
                        'A34': 'critical account'}),
    ('purpose', {'A40': 'new car', 'A41': 'used car', 'A42': 'furniture', 'A43': 'radio/television',
                 'A44': 'domestic appliances', 'A45': 'repairs', 'A46': 'education', 'A47': 'vacation',
                 'A48': 'retraining', 'A49': 'business', 'A410': 'other'}),
    ('credit_amount', None),
    ('savings', {'A61': '< 100 DM', 'A62': '100 to 500 DM', 'A63': '500 to 1000 DM', 'A64': '>= 1000 DM',
                 'A65': 'unknown or none'}),
    ('employment_since', {'A71': 'unemployed', 'A72': '< 1 year', 'A73': '1 to 4 years', 'A74': '4 to 7 years',
                          'A75': '>= 7 years'}),
    ('installment_rate', None),
    ('personal_status_sex', {'A91': 'male, divorced/separated', 'A92': 'female, divorced/separated/married',
                             'A93': 'male, single', 'A94': 'male, married/widowed', 'A95': 'female, single'}),
    ('other_debtors', {'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'}),
    ('residence_since', None),
    ('property', {'A121': 'real estate', 'A122': 'savings agreement/life insurance', 'A123': 'car or other',
                  'A124': 'unknown or none'}),
    ('age', None),
    ('other_installment_plans', {'A141': 'bank', 'A142': 'stores', 'A143': 'none'}),
    ('housing', {'A151': 'rent', 'A152': 'own', 'A153': 'for free'}),
    ('existing_credits', None),
    ('job', {'A171': 'unemployed/unskilled non-resident', 'A172': 'unskilled resident',
             'A173': 'skilled employee', 'A174': 'management/self-employed'}),
    ('people_liable', None),
    ('telephone', {'A191': 'none', 'A192': 'yes'}),
    ('foreign_worker', {'A201': 'yes', 'A202': 'no'}),
]
y = np.array(['good' if r[-1] == '1' else 'bad' for r in rows])

The numerical features take whole values;
the years of employment are ordered (`ordered=True`, in the order of their values) and
`fit_ranges` sets the minimum and maximum of every numerical feature from the data,
which bound unbounded ranges such as "more than 30.5" when drawing data points.

In [3]:
from ctreex import Feature, FeatureSpec, TreeCounterfactualExplainer

columns = []
for j, (name, categories) in enumerate(FEATURES):
    if categories is None:
        columns.append((name, None))
    else:
        present = sorted(set(r[j] for r in rows), key=lambda c: int(c[1:]))
        columns += [(name, c) for c in present]
features = [Feature(name, 'numerical' if categories is None else 'categorical',
                    display_name=name.replace('_', ' '), value_names=categories or {},
                    integer=categories is None, ordered=name == 'employment_since',
                    values=list(categories) if name == 'employment_since' else [])
            for name, categories in FEATURES]
spec = FeatureSpec(columns, features=features, separator='=')
applicants = [{name: (int(r[j]) if categories is None else r[j])
               for j, (name, categories) in enumerate(FEATURES)} for r in rows]
X = np.stack([spec.encode(a) for a in applicants])
spec.fit_ranges(X)
print(spec.features['employment_since'].values, '| age:', spec.features['age'].minimum, 'to',
      spec.features['age'].maximum)

['A71', 'A72', 'A73', 'A74', 'A75'] | age: 19.0 to 75.0


In [4]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42).fit(X, y)
explainer = TreeCounterfactualExplainer(clf, spec, class_names={'good': 'low risk', 'bad': 'high risk'})
i = int(np.flatnonzero(clf.predict(X) == 'bad')[0])
x = X[i]
explanation = explainer.explain_instance(x)
print('applicant', i, '| prediction:', explanation.prediction)
for distance, counterfactual in explanation.ranked()[:4]:
    print(distance, explainer.describe({0: counterfactual})[0])

applicant 1 | prediction: high risk
1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}
1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'other installment plans': {'is': ['none'], 'wannabe': ['bank', 'stores']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'duration': {'is': [48], 'wannabe': ['less or equal to 25.5']}}


## 1. Training statistics: Targets and rankings

Trees built from scikit-learn store the number of training samples and the impurity of every node.
`MinSamples(n)` selects leaves backed by at least `n` training samples and targets combine with `&`, `|` and `~`;
`rank_by='samples'` (`'purity'`) orders the counterfactuals at every distance by the training samples (purity) of their leaves.

In [5]:
from ctreex import MinSamples, OtherClass

context = explainer.target_context(explanation.leaf)
print('training samples per leaf:', dict(zip(context.leaf_ids.tolist(), context.leaf_samples.tolist())))
for target in (OtherClass(), OtherClass() & MinSamples(100)):
    print(target)
    for distance, counterfactual in explainer.explain_instance(x, target=target).ranked()[:3]:
        print('  ', distance, explainer.describe({0: counterfactual})[0])

training samples per leaf: {4: 91, 5: 227, 6: 31, 9: 47, 10: 42, 12: 136, 13: 32, 16: 25, 17: 39, 20: 94, 21: 22, 23: 115, 24: 99}
OtherClass()
   1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}
   1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
   2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'other installment plans': {'is': ['none'], 'wannabe': ['bank', 'stores']}}
AllOf(targets=(OtherClass(), MinSamples(n=100)))
   1 {'duration': {'is': [48], 'wannabe': ['between 11.5 and 22.5']}}
   2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'age': {'is': [22], 'wannabe': ['more than 30.5']}}


In [6]:
for rank_by in ('frequency', 'samples', 'purity'):
    print(rank_by)
    for distance, counterfactual in explainer.explain_instance(x, rank_by=rank_by).ranked()[:2]:
        print('  ', distance, explainer.describe({0: counterfactual})[0])

frequency
   1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}
   1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
samples
   1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}
   1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
purity
   1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
   1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}}


## 2. What guarantees a prediction

*Supportive explanations* are the smallest sets of features of a data point that guarantee its prediction whatever the values of the other features;
every feature is fixed at the interval between the two thresholds around its value (or its category).

In [7]:
low_risk = int(np.flatnonzero(clf.predict(X) == 'good')[0])
for supportive in explainer.supportive_explanations(X[low_risk]):
    print(explainer.describe({0: supportive.conditions})[0], '| leaves:', supportive.leaves)

{'property': {'is': ['real estate'], 'wannabe': ['real estate']}, 'duration': {'is': [6], 'wannabe': ['less or equal to 11.5']}} | leaves: [4, 16, 17, 20, 23, 24]


*Invariance ranges* list the values of every feature that keep the prediction when the other features do not change:

In [8]:
for feature, part in explainer.describe({0: explainer.invariance_ranges(x)})[0].items():
    print('{}: {}'.format(feature, part['wannabe']))

checking account: ['< 0 DM', '0 to 200 DM', '>= 200 DM']
credit history: ['no credits taken', 'all credits at this bank paid back', 'existing credits paid back so far', 'delays in the past', 'critical account']
other installment plans: ['bank', 'stores', 'none']
property: ['real estate', 'savings agreement/life insurance', 'car or other', 'unknown or none']
savings: ['< 100 DM', '100 to 500 DM', '500 to 1000 DM', '>= 1000 DM', 'unknown or none']
age: ['any value']
credit amount: ['any value']
duration: ['more than 22.5']


*Contextualised counterfactuals* add to every counterfactual the smallest sets of
the other features that guarantee its outcome whatever the remaining features --
"this change works provided that ...":

In [9]:
for c in explainer.contextualise(explanation)[:4]:
    print(c.distance, explainer.describe({0: c.change})[0], '| provided that',
          explainer.describe({0: c.context})[0] if c.context else 'nothing else')

1 {'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}} | provided that {'property': {'is': ['real estate'], 'wannabe': ['real estate']}}
1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}} | provided that nothing else
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'other installment plans': {'is': ['none'], 'wannabe': ['bank', 'stores']}} | provided that nothing else
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'duration': {'is': [48], 'wannabe': ['less or equal to 25.5']}} | provided that nothing else


## 3. Ordered features, directions and monotonicity

`directions` lets numerical and ordered features change in one direction only,
e.g., an applicant cannot get younger, cannot shorten their employment and
-- say -- cannot afford higher instalments, so the duration of the loan can only increase;
the counterfactuals asking for a shorter loan disappear:

In [10]:
directions = {'age': 'increase', 'employment_since': 'increase', 'duration': 'increase'}
for distance, counterfactual in explainer.explain_instance(x, directions=directions).ranked()[:4]:
    print(distance, explainer.describe({0: counterfactual})[0])

1 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'other installment plans': {'is': ['none'], 'wannabe': ['bank', 'stores']}}
2 {'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}, 'age': {'is': [22], 'wannabe': ['more than 30.5']}}
2 {'credit history': {'is': ['existing credits paid back so far'], 'wannabe': ['no credits taken', 'all credits at this bank paid back', 'delays in the past', 'critical account']}, 'savings': {'is': ['< 100 DM'], 'wannabe': ['100 to 500 DM', '500 to 1000 DM', '>= 1000 DM', 'unknown or none']}}


`monotonicity_violations` lists the pairs of leaves where changing a feature in a direction
-- with the other features unchanged -- makes the prediction worse,
e.g., where a shorter loan turns a low-risk applicant into a high-risk one:

In [11]:
for feature, direction in (('duration', 'decrease'), ('age', 'increase'), ('credit_amount', 'decrease')):
    violations = explainer.monotonicity_violations(feature, direction, positive='low risk')
    print(feature, direction, '->', len(violations), 'violations')
    for v in violations[:2]:
        print('   ', explainer.describe({0: v.before})[0][feature.replace('_', ' ')]['wannabe'], '->',
              explainer.describe({0: v.after})[0][feature.replace('_', ' ')]['wannabe'],
              '| when', explainer.describe({0: v.context})[0])

duration decrease -> 1 violations
    ['more than 22.5'] -> ['less or equal to 22.5'] | when {'checking account': {'is': [], 'wannabe': ['< 0 DM', '0 to 200 DM', '>= 200 DM']}, 'credit history': {'is': [], 'wannabe': ['no credits taken', 'all credits at this bank paid back', 'delays in the past', 'critical account']}, 'property': {'is': [], 'wannabe': ['unknown or none']}, 'savings': {'is': [], 'wannabe': ['100 to 500 DM', '500 to 1000 DM', '>= 1000 DM', 'unknown or none']}}
age increase -> 0 violations
credit_amount decrease -> 0 violations


## 4. Feature audits

`feature_audit` shows where a feature alone changes the prediction, e.g.,
whether personal status and gender or age decide the outcome for some applicants;
with data, it also lists the applicants whose prediction changes when the feature alone changes.
Here neither does, unlike the checking account.

In [12]:
for feature in ('personal_status_sex', 'age', 'checking_account'):
    audit = explainer.feature_audit(feature, X=X)
    print('{}: {} flips, affected leaves {}, {} of {} applicants flip with this feature alone'.format(
        feature, len(audit.flips), audit.affected_leaves, len(audit.alone), audit.n_points))

personal_status_sex: 0 flips, affected leaves [], 0 of 1000 applicants flip with this feature alone


age: 0 flips, affected leaves [], 0 of 1000 applicants flip with this feature alone


checking_account: 42 flips, affected leaves [6, 10, 12, 13, 16, 17, 20, 21, 23, 24], 381 of 1000 applicants flip with this feature alone


## 5. Instance-contrastive explanations

`explain_contrast(x, x_reference)` explains how an applicant would have to change to be treated like another one;
the counterfactuals that move towards the values of the reference come first (`reference_matches`).

In [13]:
contrast = explainer.explain_contrast(x, X[low_risk])
found = contrast.at_distance(1)
for rank, counterfactual in list(found.ranked.items())[:3]:
    print(explainer.describe({0: counterfactual})[0], '| like the reference:', found.reference_matches[rank])

{'duration': {'is': [48], 'wannabe': ['less or equal to 22.5']}} | like the reference: ['duration']
{'checking account': {'is': ['0 to 200 DM'], 'wannabe': ['no checking account']}} | like the reference: []


## 6. Data points of counterfactuals

A counterfactual describes ranges and categories;
`counterfactual_point` turns it into the nearest data point (or the middle of its ranges) and
`sample_counterfactual` draws data points at random from it (with a seed), e.g.,
to share a counterfactual without revealing the thresholds of the tree.

In [14]:
counterfactual = explanation.ranked()[0][1]
changed = list(counterfactual)
print('nearest :', {f: explainer.decode(explainer.counterfactual_point(x, counterfactual))[f] for f in changed})
print('middle  :', {f: explainer.decode(explainer.counterfactual_point(x, counterfactual, how='midpoint'))[f]
                    for f in changed})
samples = explainer.sample_counterfactual(x, counterfactual, n=5, random_state=0)
print('samples :', [{f: values[f] for f in changed} for values in explainer.decode(samples)])
print('predictions of the samples:', sorted(set(clf.predict(samples).tolist())))
print('hidden thresholds:', explainer.describe({0: counterfactual}, hide_thresholds=True, random_state=0)[0])

nearest : {'duration': 22}
middle  : {'duration': 13}
samples : [{'duration': 13}, {'duration': 9}, {'duration': 7}, {'duration': 19}, {'duration': 22}]
predictions of the samples: ['good']
hidden thresholds: {'duration': {'is': [48], 'wannabe': [13]}}


`exemplars` returns real applicants from the leaves a counterfactual leads to (beware that they reveal the training data):

In [15]:
leaves = explainer.counterfactual_leaves(x, counterfactual)
for values in explainer.decode(explainer.exemplars(X, leaves, n=2, random_state=0)):
    print({f: values[f] for f in ('duration', 'credit_amount', 'checking_account', 'age')})

{'duration': 12, 'credit_amount': 741, 'checking_account': 'A11', 'age': 22}
{'duration': 18, 'credit_amount': 1345, 'checking_account': 'A11', 'age': 26}


## 7. Plain-text explanations

`explanation_text` and `supportive_text` turn the explanations into sentences from
a generic template built from the display names of the features and the names of their categories and classes.

In [16]:
for sentence in explainer.explanation_text(explanation, n=3):
    print(sentence)
print()
for sentence in explainer.explanation_text(explanation, explainer.contextualise(explanation), n=2):
    print(sentence)
print()
for sentence in explainer.supportive_text(X[low_risk]):
    print(sentence)

The prediction is high risk.
Had duration been less or equal to 22.5 (instead of 48), the prediction would have been low risk.
Had checking account been no checking account (instead of 0 to 200 DM), the prediction would have been low risk.
Had checking account been no checking account (instead of 0 to 200 DM) and other installment plans been bank or stores (instead of none), the prediction would have been low risk.

The prediction is high risk.
Had duration been less or equal to 22.5 (instead of 48), the prediction would have been low risk, provided that property is real estate.
Had checking account been no checking account (instead of 0 to 200 DM), the prediction would have been low risk.

The prediction is low risk since property is real estate and duration is less or equal to 11.5, whatever the other features.
